In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [41]:
# Connect to the SQLite database
conn = sqlite3.connect('../data/movies.db')
# Load movies data from the database
df = pd.read_sql_query("SELECT * FROM movies", conn)

In [42]:
df.head()

,movie_id,title,year,genres,rating
0,1,Toy Story,1995,Adventure|Animation|Children|Comedy|Fantasy,4
1,2,Jumanji,1995,Adventure|Children|Fantasy,3
2,3,Grumpier Old Men,1995,Comedy|Romance,3
3,4,Waiting to Exhale,1995,Comedy|Drama|Romance,2
4,5,Father of the Bride Part II,1995,Comedy,3


In [43]:
df['genres'] = df['genres'].str.split("|")
df.head()

,movie_id,title,year,genres,rating
0,1,Toy Story,1995,"[Adventure, Animation, Children, Comedy, Fantasy]",4
1,2,Jumanji,1995,"[Adventure, Children, Fantasy]",3
2,3,Grumpier Old Men,1995,"[Comedy, Romance]",3
3,4,Waiting to Exhale,1995,"[Comedy, Drama, Romance]",2
4,5,Father of the Bride Part II,1995,[Comedy],3


In [44]:
df['genre_str'] = df['genres'].apply(lambda x: ' '.join(x))

In [67]:
from sklearn.feature_extraction.text import CountVectorizer

# Define a named function instead of a lambda
def space_tokenizer(text):
    return text.split()

# Use the named function in the vectorizer
vectorizer = CountVectorizer(
    tokenizer=space_tokenizer,  
    binary=True,                     
    lowercase=False                  
)

genre_matrix = vectorizer.fit_transform(df['genre_str'])


c:\Users\Beka\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [68]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(genre_matrix)

In [69]:
# Step 3: RECOMMENDATION FUNCTION
class GenreRecommender:
    def __init__(self, vectorizer, similarity_matrix, df):
        self.vectorizer = vectorizer
        self.similarity_matrix = similarity_matrix
        self.df = df.copy()
        self.indices = pd.Series(df.index, index=df['title']).to_dict()
        self.titles = df['title'].tolist()
        
    def get_recommendations(self, title, top_n=5, exclude_self=True):
        """
        Get movie recommendations based on genre similarity
        
        Parameters:
        -----------
        title : str
            Movie title to base recommendations on
        top_n : int
            Number of recommendations to return
        exclude_self : bool
            Whether to exclude the input movie from results
            
        Returns:
        --------
        DataFrame with recommendations and similarity scores
        """
        # Validate input
        if title not in self.indices:
            available = [t for t in self.titles if title.lower() in t.lower()]
            if available:
                return f"Movie '{title}' not found. Did you mean: {available}?"
            return f"Movie '{title}' not found in database."
        
        # Get index of movie
        idx = self.indices[title]
        
        # Get similarity scores for all movies
        sim_scores = list(enumerate(self.similarity_matrix[idx]))
        
        # Sort by similarity (descending)
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Exclude self if requested
        if exclude_self:
            sim_scores = sim_scores[1:top_n+1]
        else:
            sim_scores = sim_scores[:top_n]
        
        # Get movie indices
        movie_indices = [i[0] for i in sim_scores]
        scores = [i[1] for i in sim_scores]
        
        # Create results DataFrame
        recommendations = self.df.iloc[movie_indices][['movie_id', 'title', 'genres', 'rating']].copy()
        recommendations['similarity_score'] = scores
        recommendations['rank'] = range(1, len(recommendations) + 1)
        
        # Reorder columns
        recommendations = recommendations[['rank', 'movie_id', 'title', 'genres', 'rating', 'similarity_score']]
        
        return recommendations
    
    def get_batch_recommendations(self, titles, top_n=5):
        """Get recommendations for multiple movies and aggregate"""
        all_recs = []
        for title in titles:
            rec = self.get_recommendations(title, top_n=top_n)
            if isinstance(rec, pd.DataFrame):
                all_recs.append(rec)
        
        if not all_recs:
            return "No valid movies found."
        
        combined = pd.concat(all_recs)
        combined['genres'] = combined['genres'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
        # Group by movie and average similarity scores
        aggregated = combined.groupby(['movie_id', 'title', 'genres', 'rating'])['similarity_score'].mean().reset_index()
        aggregated = aggregated.sort_values('similarity_score', ascending=False).head(top_n)
        aggregated['rank'] = range(1, len(aggregated) + 1)
        
        return aggregated[['rank', 'movie_id', 'title', 'genres', 'rating', 'similarity_score']]
    
    def get_similarity_between(self, title1, title2):
        """Get similarity score between two specific movies"""
        if title1 not in self.indices or title2 not in self.indices:
            return "One or both movies not found."
        
        idx1 = self.indices[title1]
        idx2 = self.indices[title2]
        score = self.similarity_matrix[idx1, idx2]
        
        return {
            'movie_1': title1,
            'movie_2': title2,
            'similarity_score': round(score, 4),
            'similarity_percentage': f"{score*100:.1f}%"
        }

In [70]:
# Initialize recommender
recommender = GenreRecommender(vectorizer, cosine_sim, df)

# Test single movie recommendation
print("\nSingle Movie Recommendation ('Heat'):")
result = recommender.get_recommendations('Heat', top_n=5)
print(result.to_string(index=False))


Single Movie Recommendation ('Heat'):
 rank  movie_id                      title                    genres  rating  similarity_score
    1        23                  Assassins [Action, Crime, Thriller]       3               1.0
    2       165 Die Hard: With a Vengeance [Action, Crime, Thriller]       4               1.0
    3       185                   Net, The [Action, Crime, Thriller]       3               1.0
    4       288       Natural Born Killers [Action, Crime, Thriller]       3               1.0
    5       479             Judgment Night [Action, Crime, Thriller]       2               1.0


In [71]:
# Test similarity between two movies
print(f"\nSimilarity Check:")
sim_result = recommender.get_similarity_between('Toy Story', 'Balto')
print(f"  {sim_result['movie_1']} ↔ {sim_result['movie_2']}: {sim_result['similarity_score']}")


Similarity Check:
  Toy Story ↔ Balto: 0.7746


In [72]:
# Test batch recommendation
print(f"\nBatch Recommendation (['Toy Story', 'Jumanji']):")
batch_result = recommender.get_batch_recommendations(['Toy Story', 'Jumanji'], top_n=5)
print(batch_result.to_string(index=False))


Batch Recommendation (['Toy Story', 'Jumanji']):
 rank  movie_id                              title                       genres  rating  similarity_score
    1        60        Indian in the Cupboard, The Adventure, Children, Fantasy       3               1.0
    2       126         NeverEnding Story III, The Adventure, Children, Fantasy       2               1.0
    3      1009           Escape to Witch Mountain Adventure, Children, Fantasy       3               1.0
    4      2043 Darby O'Gill and the Little People Adventure, Children, Fantasy       3               1.0
    5      2093                       Return to Oz Adventure, Children, Fantasy       3               1.0


In [ ]:
from datetime import datetime
import json
import os
import pickle

output_dir = '../artifacts'
os.makedirs(output_dir, exist_ok=True)


# Save the fitted vectorizer
vectorizer_path = f'{output_dir}/count_vectorizer.pkl'
with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)
print(f"✓ Vectorizer saved: {vectorizer_path}")

# Save the similarity matrix
similarity_path = f'{output_dir}/similarity_matrix.pkl'
with open(similarity_path, 'wb') as f:
    pickle.dump(cosine_sim, f)
print(f"✓ Similarity matrix saved: {similarity_path}")

# Save the recommender class instance
recommender_path = f'{output_dir}/recommender.pkl'
with open(recommender_path, 'wb') as f:
    pickle.dump(recommender, f)
print(f"✓ Recommender instance saved: {recommender_path}")

# Save the dataframe
df_path = f'{output_dir}/movies_data.pkl'
with open(df_path, 'wb') as f:
    pickle.dump(df, f)
print(f"✓ Movie data saved: {df_path}")

# Save metadata (JSON for human readability)
metadata = {
    'created_at': datetime.now().isoformat(),
    'model_type': 'Content-Based (CountVectorizer + Cosine Similarity)',
    'num_movies': len(df),
    'num_genres': len(vectorizer.get_feature_names_out()),
    'genres': vectorizer.get_feature_names_out().tolist(),
    'vectorizer_config': {
        'type': 'CountVectorizer',
        'binary': True,
        'lowercase': False
    },
    'similarity_metric': 'cosine',
    'files': {
        'vectorizer': 'count_vectorizer.pkl',
        'similarity_matrix': 'similarity_matrix.pkl',
        'recommender': 'recommender.pkl',
        'data': 'movies_data.pkl'
    }
}

metadata_path = f'{output_dir}/metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved: {metadata_path}")

# Save as compressed numpy array (efficient for large matrices)
npz_path = f'{output_dir}/similarity_matrix.npz'
np.savez_compressed(npz_path, similarity=cosine_sim)
print(f"✓ Compressed similarity matrix saved: {npz_path}")

print(f"\n📦 All model artifacts saved to: {output_dir}")
print(f"   Total size: {sum(os.path.getsize(os.path.join(output_dir, f)) for f in os.listdir(output_dir)) / 1024:.2f} KB")

✓ Vectorizer saved: ../artifacts/count_vectorizer.pkl
✓ Similarity matrix saved: ../artifacts/similarity_matrix.pkl
✓ Recommender instance saved: ../artifacts/recommender.pkl
✓ Movie data saved: ../artifacts/movies_data.pkl
✓ Metadata saved: ../artifacts/metadata.json
✓ Compressed similarity matrix saved: ../artifacts/similarity_matrix.npz

📦 All model artifacts saved to: ../artifacts
   Total size: 1535817.50 KB
